# 深度學習框架應用（大學部）｜主題 15：模型部署：ONNX、TorchScript 與 API 服務化

這本筆記本配合投影片《主題 15：模型部署》，先訓練一個小小的 Fashion-MNIST 模型，再一路把它「帶出門」：存檔與讀檔、轉成 TorchScript、匯出 ONNX 交給 ONNX Runtime、包成 `/predict` 網路服務，最後量延遲。**練習一～五，每題 20%，合計 100%**。

| 練習 | 題目 | 投影片（練習頁） | 觀念在哪一頁 |
|---|---|---|---|
| 一 | 存權重、讀回來、存 checkpoint | 第 107 頁 | 第 14～22 頁 |
| 二 | TorchScript：trace 和 script，遇到 if 誰會出錯 | 第 108 頁 | 第 23～32 頁 |
| 三 | 匯出 ONNX，用 ONNX Runtime 推論並比對（動態 batch 軸） | 第 109 頁 | 第 33～48 頁 |
| 四 | 用 FastAPI 包成 /predict 服務，用 TestClient 測五種情況 | 第 110 頁 | 第 53～72 頁 |
| 五 | 批次推論與延遲量測（P50、P99、HTTP 開銷） | 第 111 頁 | 第 73～85 頁 |

**怎麼用**

1. 用私人 Google 帳號登入（不使用學校帳號），按上方的「複製到雲端硬碟」（或「檔案 › 在雲端硬碟中儲存複本」）另存一份自己的複本，之後都改自己那份。**沒有另存就直接寫，分頁一關就全部不見。** 存好後先把檔名改成 `深度學習_W15_學號_姓名.ipynb`（左上角點檔名就能改）。
2. 先跑「環境準備」（會安裝 onnx、onnxruntime、FastAPI 等套件），再跑「準備：訓練一個小模型」，然後由上往下做。每一題只有一格標 ✏️ 的程式格：把每個 `___` 換成答案再執行，提示就寫在同一行後面的註解。還沒把 `___` 換掉就執行，會出現紅字錯誤（例如 AttributeError、TypeError），錯誤訊息裡箭頭 `---->` 指的那一行就是還沒填的空格，填好再跑一次就好。
3. **這本全部在 CPU 上跑，不用開 GPU**：部署的機器多半只有 CPU，而且延遲要在同一種硬體上比才有意義。Fashion-MNIST 第一次執行會自動下載（約 30 MB）；只拿 12000 張訓練、2000 張測試，CPU 約 1 分鐘練完。每台電腦量到的時間不一樣，練習五的數字和投影片不同是正常的，看「誰比誰快、快幾倍」就好。
4. 練習四用 FastAPI 的 `TestClient` 直接在筆記本裡送請求，**不用開伺服器**；想在自己電腦上用 uvicorn 真的跑起來，照投影片「在電腦教室本機執行的路徑」那頁做（不計分）。
5. **每題做完就截一張整個螢幕的圖（截圖 ①～⑤）**，要看得到身份指紋：左上的檔名、右上的 Google 頭貼、右下工作列的日期時間，以及這一題的輸出。**看不到身份指紋，那一題不計分。**

**繳交**：做完照最後一格「繳交前檢查」，把**分享連結 1 個、.ipynb 檔 1 個、截圖 5 張（①～⑤）**交到 Flipclass 作業區。

## 環境準備

第一行的 `%pip install` 會裝好這次要用的套件：`onnx`（讀寫 ONNX 檔）、`onnxscript`（新版 PyTorch 匯出 ONNX 時要用）、`onnxruntime`（跑 ONNX 模型）、`fastapi` 和 `httpx`（寫服務、在筆記本裡測服務）。

In [ ]:
# ===== 環境準備（直接執行，第一次約 1 分鐘）=====
%pip install -q onnx onnxscript onnxruntime fastapi httpx
import os, json, time, warnings
import numpy as np
import torch
import torch.nn as nn
from torchvision import datasets
import onnx, onnxruntime as ort
from fastapi import FastAPI, HTTPException
from fastapi.testclient import TestClient
from pydantic import BaseModel
warnings.filterwarnings('ignore', category=FutureWarning)   # torch.jit 會提醒「已不建議使用」，先藏起來
print('PyTorch', torch.__version__, '｜onnx', onnx.__version__, '｜onnxruntime', ort.__version__)

## 準備：訓練一個要「帶出門」的小模型

這次的主角不是訓練，所以只練一個小模型：攤平 → 784→256 → BatchNorm → ReLU → Dropout → 256→128 → ReLU → 128→10。有 BatchNorm 和 Dropout，是為了讓你看到 `eval()` 和「緩衝」在部署時為什麼重要。直接執行，不用改。

| 編號 | 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9 |
|---|---|---|---|---|---|---|---|---|---|---|
| 英文（程式裡用） | T-shirt | Trouser | Pullover | Dress | Coat | Sandal | Shirt | Sneaker | Bag | Boot |
| 中文 | T 恤 | 褲子 | 套頭毛衣 | 洋裝 | 外套 | 涼鞋 | 襯衫 | 運動鞋 | 包包 | 短靴 |

In [ ]:
# ===== 準備：載入 Fashion-MNIST，訓練一個小模型（直接執行，CPU 約 1 分鐘）=====
train = datasets.FashionMNIST('data', train=True, download=True)
test = datasets.FashionMNIST('data', train=False, download=True)
Xtr = train.data[:12000].float().div(255).unsqueeze(1)   # 訓練 12000 張，形狀 (12000, 1, 28, 28)，0～1
ytr = train.targets[:12000]
Xte = test.data[:2000].float().div(255).unsqueeze(1)     # 測試 2000 張
yte = test.targets[:2000]
NAMES = ['T-shirt', 'Trouser', 'Pullover', 'Dress', 'Coat',
         'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Boot']        # 0～9 類
HP = {'h1': 256, 'h2': 128, 'drop': 0.2}                     # 超參數：跟權重一起存，讀檔時才蓋得出一樣的架子

def make_model(h1, h2, drop):
    return nn.Sequential(nn.Flatten(), nn.Linear(784, h1), nn.BatchNorm1d(h1), nn.ReLU(),
                         nn.Dropout(drop), nn.Linear(h1, h2), nn.ReLU(), nn.Linear(h2, 10))

torch.manual_seed(42)                                        # 固定亂數，大家練出來差不多
model = make_model(**HP)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()
EPOCHS = 3
for ep in range(1, EPOCHS + 1):
    model.train()
    perm = torch.randperm(len(Xtr))
    for i in range(0, len(Xtr), 128):                        # 一批 128 張
        idx = perm[i:i + 128]
        opt.zero_grad(); loss = loss_fn(model(Xtr[idx]), ytr[idx]); loss.backward(); opt.step()
    model.eval()
    with torch.no_grad():
        acc = (model(Xte).argmax(1) == yte).float().mean().item()
    print(f'第 {ep} 回合：測試準確率 {acc:.3f}')
print('model 已經切成 eval()：Dropout 關掉、BatchNorm 改用存好的平均與變異數')

## 練習一：存權重、讀回來、存 checkpoint (20%)

> 題目在投影片**第 107 頁**｜觀念在第 14～22 頁

**情境**：小智的衣服辨識模型練好了，要交給隔壁組的同學用。Géron 建議只存權重（`state_dict`），讀的時候先用同一組超參數蓋出一樣的架子，再把權重放進去；想接著練，就連最佳化器和回合數一起存成 checkpoint。

**這題在練**：數參數、存 `state_dict`、用 `weights_only=True` 安全地讀回來、確認輸出一模一樣，再存一個 checkpoint

**步驟**

1. 用 `p.___()` 數每個參數張量有幾個元素，加起來就是參數個數；再看 `state_dict` 有幾個張量（比參數張量多幾個？）
2. 用 `torch.save(model.___(), …)` 只存權重，看檔案多大；用 `make_model(**HP)` 蓋一個新架子，`load_state_dict` 放進權重、切 `eval()`，比較兩個模型的輸出
3. 把權重、最佳化器、回合數、超參數包成一個字典存成 checkpoint，再讀回來看裡面有哪些鍵

**要繳交**

- 截圖 ①：整個螢幕，看得到身份指紋和這一題的輸出
- 文字題：寫在下面的「✏️ 文字回答」

In [ ]:
# ✏️ 練習一：存權重、讀回來，再存一個 checkpoint
n = sum(p.___() for p in model.parameters())  # ✏️ 個數
sd = model.___()               # ✏️ 權重字典
torch.save(sd, 'fmnist_w.pt')
print(f'參數 {n} 個｜state_dict {len(sd)} 個張量｜'
      f'檔案 {os.path.getsize("fmnist_w.pt")} bytes')
new = make_model(**HP).___()   # ✏️ 推論模式
w = torch.___('fmnist_w.pt', weights_only=True)  # ✏️ 讀
new.load_state_dict(w)         # 權重放進架子
print('相同？', torch.equal(model(Xte[:8]), new(Xte[:8])))
ck = {'model': sd, 'opt': opt.state_dict(), 'hp': HP}
torch.save(ck, 'fmnist_ckpt.pt')
back = torch.load('fmnist_ckpt.pt', weights_only=True)
print('checkpoint：', list(back), back['hp'])

#### 📸 截圖 ①

跑完上面的程式格，截一張**整個螢幕**的圖，畫面要同時看得到：

- **身份指紋**：左上的檔名、右上的 Google 頭貼、右下工作列的日期時間
- **這一題的輸出**：參數個數與 state_dict 張量數、檔案大小、「輸出一模一樣？ True」、checkpoint 裡的鍵

看不到身份指紋，這一題不計分。

#### ✏️ 文字回答

**1. 參數張量只有 8 個，state_dict 卻有 11 個。多出來的 3 個是什麼？為什麼也要存？**

（在這裡寫）

**2. 為什麼讀檔要加 `weights_only=True`？直接 `torch.save(model)` 存整個模型物件有哪兩個問題？**

（在這裡寫）

## 練習二：TorchScript：trace 和 script，遇到 if 誰會出錯 (20%)

> 題目在投影片**第 108 頁**｜觀念在第 23～32 頁

**情境**：寫 C++ 的同事想直接載入模型，所以要轉成 TorchScript。轉法有兩種：`trace` 拿一筆範例跑一次、把走過的運算錄下來；`script` 直接讀 `forward` 的原始碼來轉。先用一個 `forward` 裡有 if 的小模型 `Gate` 試試看，再把練好的模型轉成 TorchScript 存檔。

**這題在練**：分辨 trace 和 script；從 `tr.code` 看出 trace 把 if 寫死了；把練好的模型存成 TorchScript 檔，讀回來比對

**步驟**

1. 先執行「練習二：準備」，看 `Gate` 在總和正、負時各回什麼
2. 填好 `torch.jit.___`（錄影的是哪一個、讀原始碼的是哪一個），用總和是負的 `[-2., -5.]` 去考三個版本，再印出 `tr.code`
3. 把練好的模型 trace 成 TorchScript、`___` 存成檔，讀回來和原模型比最大誤差

> 新版 PyTorch 會提醒 `torch.jit` 已不建議使用（改用 `torch.compile`、`torch.export`），環境準備那格已經先藏起來；功能照常。執行時看到的 **TracerWarning** 是故意留著的：它在提醒你 trace 遇到了 if。

**要繳交**

- 截圖 ②：整個螢幕，看得到身份指紋和這一題的輸出
- 文字題：寫在下面的「✏️ 文字回答」

In [ ]:
# ===== 練習二：準備（直接執行）=====
class Gate(nn.Module):
    """forward 裡有 if：總和大於 0 就乘 2，否則乘 -1"""
    def forward(self, x):
        if x.sum() > 0:
            return x * 2
        return x * -1

print('總和是正的：', Gate()(torch.tensor([3., 1.])))
print('總和是負的：', Gate()(torch.tensor([-2., -5.])))

In [ ]:
# ✏️ 練習二：trace 和 script，遇到 if 誰會出錯
g = Gate()                            # forward 裡有 if
tr = torch.jit.___(g, torch.tensor([3., 1.]))  # ✏️ 錄影
sc = torch.jit.___(g)                 # ✏️ 讀原始碼
x = torch.tensor([-2., -5.])          # 總和是負的
print('原本  ：', g(x))
print('trace ：', tr(x))
print('script：', sc(x))
print(tr.code)                        # 錄到的程式長怎樣
ts = torch.jit.trace(model, Xte[:1])  # 模型沒有 if
ts.___('fmnist_ts.pt')                # ✏️ 存成檔
back = torch.jit.load('fmnist_ts.pt')
d = (back(Xte[:64]) - model(Xte[:64])).abs().max()
print('TorchScript 和原模型的最大誤差：', d.item())

#### 📸 截圖 ②

跑完上面的程式格，截一張**整個螢幕**的圖，畫面要同時看得到：

- **身份指紋**：左上的檔名、右上的 Google 頭貼、右下工作列的日期時間
- **這一題的輸出**：原本、trace、script 三行，`tr.code` 印出來的程式，以及最大誤差那一行

看不到身份指紋，這一題不計分。

#### ✏️ 文字回答

**1. trace 版為什麼答成 `[-4., -10.]`？從印出來的 `tr.code` 看得出什麼？**

（在這裡寫）

**2. 練好的 Fashion 模型用 trace 為什麼沒問題？如果 forward 裡有「看輸入內容決定」的 if，要用哪一種？**

（在這裡寫）

## 練習三：匯出 ONNX，用 ONNX Runtime 推論並比對 (20%)

> 題目在投影片**第 109 頁**｜觀念在第 33～48 頁

**情境**：產線的控制程式是 C# 寫的、邊緣盒子上沒有 PyTorch，所以把模型匯出成跨框架的 ONNX，交給 ONNX Runtime 執行。匯出以後一定要做數值比對：同一批輸入，ONNX Runtime 和 PyTorch 的輸出差距要小於 1e-4 才算轉檔成功。

**這題在練**：`torch.onnx.export` 設定輸入輸出名字與動態 batch 軸、`onnx.checker` 結構檢查、建立 `InferenceSession`、送 float32 的 numpy 陣列、和 PyTorch 比最大誤差

**步驟**

1. 先執行「練習三：準備」（`CPU` 和 `torch_out` 兩個小工具）
2. 在 ✏️ 格填好匯出函式、建立 session 的類別、float32；看輸入形狀的第 0 維是不是 `'batch'`，再用 1、8、64 張各跑一次
3. 執行「練習三之二」：看計算圖有哪些算子、檔案多大，以及「沒設動態軸」和「送 float64」各會看到什麼錯誤

> 新版 PyTorch 匯出時會印出好幾行 `[torch.onnx] …` 進度，也可能提醒 dynamic_axes 建議改用 dynamic_shapes，都是正常的。

**要繳交**

- 截圖 ③：整個螢幕，看得到身份指紋、✏️ 格的輸出和「練習三之二」的輸出
- 文字題：寫在下面的「✏️ 文字回答」

In [ ]:
# ===== 練習三：準備（直接執行）=====
CPU = ['CPUExecutionProvider']             # ONNX Runtime 用 CPU 算
ONNX = 'fmnist.onnx'                       # 匯出的檔名

@torch.no_grad()
def torch_out(x):
    """拿 numpy 陣列給 PyTorch 模型算，回傳 numpy，方便和 ONNX Runtime 的輸出相減"""
    return model(torch.from_numpy(x)).numpy()

print('PyTorch 模型的輸出形狀：', torch_out(Xte[:3].numpy()).shape)

In [ ]:
# ✏️ 練習三：匯出 ONNX，用 ONNX Runtime 推論並比對
torch.onnx.___(model, Xte[:1], ONNX,        # ✏️ 匯出
    input_names=['input'], output_names=['logits'],
    dynamic_axes={'input': {0: 'batch'},    # 第 0 維可變
                  'logits': {0: 'batch'}},
    opset_version=18)
onnx.checker.check_model(onnx.load(ONNX))   # 結構檢查
sess = ort.___(ONNX, providers=CPU)         # ✏️
print('模型要的輸入形狀：', sess.get_inputs()[0].shape)
for b in (1, 8, 64):
    x = Xte[:b].numpy().astype(np.___)      # ✏️ 32 位元
    out = sess.run(None, {'input': x})[0]
    err = np.abs(out - torch_out(x)).max()
    print(f'{b:2d} 張 → {out.shape}，最大誤差 {err:.1e}')

In [ ]:
# 練習三之二（直接執行）：看計算圖、看檔案，再看「沒設動態軸」會怎樣
print('計算圖的算子：', [n.op_type for n in onnx.load('fmnist.onnx').graph.node])
print('檔案大小：', {f: os.path.getsize(f) for f in sorted(os.listdir('.')) if f.startswith('fmnist.onnx')})
torch.onnx.export(model, Xte[:1], 'fixed.onnx', input_names=['input'],
                  output_names=['logits'], opset_version=18)    # 故意不設 dynamic_axes
s2 = ort.InferenceSession('fixed.onnx', providers=CPU)
print('沒設動態軸，輸入形狀寫死成：', s2.get_inputs()[0].shape)
try:
    s2.run(None, {'input': Xte[:4].numpy()})
except Exception as e:
    print('送 4 張進去 →', str(e))
try:
    sess.run(None, {'input': Xte[:4].numpy().astype(np.float64)})
except Exception as e:
    print('送 float64 進去 →', str(e))

#### 📸 截圖 ③

跑完上面的程式格，截一張**整個螢幕**的圖，畫面要同時看得到：

- **身份指紋**：左上的檔名、右上的 Google 頭貼、右下工作列的日期時間
- **這一題的輸出**：輸入形狀、1／8／64 張的輸出形狀與最大誤差，以及「練習三之二」印出的算子、檔案大小和兩段錯誤訊息

看不到身份指紋，這一題不計分。

#### ✏️ 文字回答

**1. 三次的最大誤差各是多少？為什麼不會剛好是 0？小於多少才算轉檔成功？**

（在這裡寫）

**2. 拿掉 dynamic_axes 之後送 4 張會怎樣？送 float64 又會怎樣？服務端為什麼一定要設動態軸？**

（在這裡寫）

## 練習四：用 FastAPI 包成 /predict 服務 (20%)

> 題目在投影片**第 110 頁**｜觀念在第 53～72 頁

**情境**：前端同學只想送一組 JSON 過來、拿回答案，不想裝 PyTorch。把練習三的 ONNX 模型包成一個 `/predict` 網路服務：收 784 個數字，回傳類別編號、名稱、機率和模型版本；送錯就回 400。

**這題在練**：FastAPI 的路由（`@app.post`）、用 pydantic 的 `Req` 規定本文格式、長度檢查回 400、把 logits 做 softmax 變成機率；再用 `TestClient` 不開伺服器就送請求，看 200、400、422、405 各在什麼時候出現

**步驟**

1. 先執行「練習四：準備」：`VER`（模型版本）和 `Req`（本文要有 `pixels`，是一串浮點數）
2. 在 ✏️ 格填好：送資料用哪個 HTTP 方法、一張圖幾個數字、送錯回哪個狀態碼、機率最大的位置
3. 執行「練習四之二」：加上 `/health`，用 `TestClient` 測五種情況，對一對每一種的狀態碼

> 注意 `sess`（ONNX Runtime 的 session）是在練習三建好、放在函式外面的：**整個服務只載入一次**，不要寫進 `predict()` 裡面。

**要繳交**

- 截圖 ④：整個螢幕，看得到身份指紋和「練習四之二」印出的五行
- 文字題：寫在下面的「✏️ 文字回答」

In [ ]:
# ===== 練習四：準備（直接執行）=====
VER = '1.0'                           # 模型版本：回應裡一起帶回去，出事才查得到是哪一版

class Req(BaseModel):
    pixels: list[float]               # 784 個 0～1 的數字；型別不對，FastAPI 會自動回 422

print('請求的本文長這樣：', json.dumps({'pixels': [0.0, 0.5, 1.0]}), '…（共 784 個）')

In [ ]:
# ✏️ 練習四：把模型包成 /predict 服務
app = FastAPI(title='Fashion API', version=VER)  # 重建
@app.___('/predict')          # ✏️ 送資料用哪個方法
def predict(r: Req):
    if len(r.pixels) != ___:  # ✏️ 一張圖幾個數字
        raise HTTPException(___, '要 784 個')  # ✏️ 狀態碼
    x = np.float32(r.pixels).reshape(1, 1, 28, 28)
    z = sess.run(None, {'input': x})[0][0]  # 10 個 logits
    p = np.exp(z - z.max()); p = p / p.sum()  # softmax
    k = int(p.___())          # ✏️ 最大的位置
    return {'label': k, 'name': NAMES[k],
            'prob': round(float(p[k]), 4), 'ver': VER}

In [ ]:
# 練習四之二（直接執行）：加上 /health，再用 TestClient 測五種情況（不用開伺服器）
ZERO = np.zeros((1, 1, 28, 28), np.float32)

@app.get('/health')
def health():                         # 真的跑一次輕量推論，確認模型還活著
    sess.run(None, {'input': ZERO})
    return {'status': 'ok', 'ver': VER}

client = TestClient(app, raise_server_exceptions=False)
img = Xte[0].ravel().tolist()         # 第 0 張測試圖攤平成 784 個數字
tests = [('正常送 784 個', 'post', '/predict', {'pixels': img}),
         ('只送 783 個', 'post', '/predict', {'pixels': img[:783]}),
         ('pixels 放 "abc"', 'post', '/predict', {'pixels': 'abc'}),
         ('用 GET 叫 /predict', 'get', '/predict', None),
         ('健康檢查', 'get', '/health', None)]
codes = []
for name, method, path, body in tests:
    r = client.post(path, json=body) if method == 'post' else client.get(path)
    codes.append(r.status_code)
    print(f'{name:14s} → {r.status_code}  {r.text[:72]}')
print('第 0 張的正確答案：', NAMES[yte[0]])

#### 📸 截圖 ④

跑完上面的程式格，截一張**整個螢幕**的圖，畫面要同時看得到：

- **身份指紋**：左上的檔名、右上的 Google 頭貼、右下工作列的日期時間
- **這一題的輸出**：「練習四之二」印出的五行狀態碼與回應，以及第 0 張的正確答案

看不到身份指紋，這一題不計分。

#### ✏️ 文字回答

**1. 送 783 個數字回 400，pixels 放 "abc" 回 422，用 GET 叫 /predict 回 405。這三個各是誰擋下來的？**

（在這裡寫）

**2. 為什麼 `sess` 要在函式外面建一次？如果寫進 predict() 裡面會怎樣？**

（在這裡寫）

## 練習五：批次推論與延遲量測 (20%)

> 題目在投影片**第 111 頁**｜觀念在第 73～85 頁

**情境**：服務要上線了，先量量看有多快。延遲每次量都會飄，所以照 NLPT 第 8 章的做法：先暖機幾次不計時，再量很多次，看分布而不是只看一次。這題比較 PyTorch 和 ONNX Runtime、一次 1 張和一次 32 張，最後再量「經過 HTTP」的一整趟。

**這題在練**：用 `np.percentile` 算 P50（中位數）和 P99；算出批次推論平均每張花多少；比較直接呼叫模型和經過 HTTP 差多少

**步驟**

1. 先執行「練習五：準備」：PyTorch 和 ONNX Runtime 都改成只用 1 條執行緒（Colab 只有 2 個核心，兩邊的執行緒會互搶，量出來的數字會亂跳）；`bench()` 會先暖機 10 次，再量 200 次，回傳每一次的毫秒數
2. 在 ✏️ 格填好：一批幾張、P50 和 P99 要給 `np.percentile` 什麼數字、平均每張要除以什麼
3. 看輸出：哪一種最快？一次 32 張平均每張快了幾倍？經過 HTTP 一整趟是直接呼叫的幾倍？

> 每台電腦、每一次量到的數字都不一樣；P99 偶爾會因為電腦在忙別的事而特別大，這正是要看 P99 的原因。

**要繳交**

- 截圖 ⑤：整個螢幕，看得到身份指紋和這一題的五行輸出
- 文字題：寫在下面的「✏️ 文字回答」

In [ ]:
# ===== 練習五：準備（直接執行）=====
torch.set_num_threads(1)                        # 公平比較：PyTorch 和 ONNX Runtime 都只用 1 條執行緒
so = ort.SessionOptions()
so.intra_op_num_threads = 1
sess = ort.InferenceSession(ONNX, so, providers=CPU)   # 重建 session（/predict 也會改用這一個）

@torch.no_grad()
def bench(fn, x, n=200, warm=10):
    """先暖機 warm 次（不計時），再量 n 次；回傳每一次花了幾 ms（numpy 陣列）"""
    for _ in range(warm):
        fn(x)
    t = []
    for _ in range(n):
        t0 = time.perf_counter()
        fn(x)
        t.append((time.perf_counter() - t0) * 1000)
    return np.array(t)

def http_call(v):
    """經過 HTTP 送一次請求：打包 JSON → FastAPI 解析檢查 → 推論 → 回傳 JSON"""
    return client.post('/predict', json={'pixels': v})

print('CPU 核心數：', os.cpu_count(), '｜PyTorch 執行緒：', torch.get_num_threads(), '｜下面的時間單位都是 ms')

In [ ]:
# ✏️ 練習五：一次 1 張和一次 32 張，量 P50、P99
x1, x32 = Xte[:1].numpy(), Xte[:___].numpy()  # ✏️ 一批
runs = [('PyTorch', lambda v: model(torch.from_numpy(v))),
        ('ORT', lambda v: sess.run(None, {'input': v}))]
for name, fn in runs:
    for x in (x1, x32):
        t = bench(fn, x)          # 暖機 10、量 200 次
        p50, p99 = np.percentile(t, [___, ___])  # ✏️
        each = p50 / len(___)     # ✏️ 平均每張
        print(f'{name:8s}{len(x):3d} 張  P50 {p50:.3f}',
              f'P99 {p99:.3f}  每張 {each:.4f}')
img = Xte[0].ravel().tolist()
t = bench(http_call, img, n=100)  # 送 100 次
print(f'經過 HTTP 一整趟：P50 {np.median(t):.2f} ms')

#### 📸 截圖 ⑤

跑完上面的程式格，截一張**整個螢幕**的圖，畫面要同時看得到：

- **身份指紋**：左上的檔名、右上的 Google 頭貼、右下工作列的日期時間
- **這一題的輸出**：PyTorch、ORT 各兩行（1 張、32 張）的 P50、P99、每張，以及「經過 HTTP」那一行

看不到身份指紋，這一題不計分。

#### ✏️ 文字回答

**1. 一次 32 張，平均每張比一次 1 張快了幾倍？那服務為什麼不一律湊滿 32 張再算？**

（在這裡寫）

**2. 經過 HTTP（TestClient）的 P50 大約是直接呼叫 ORT 的幾倍？多出來的時間花在哪裡？這代表什麼？**

（在這裡寫）

## AI 使用紀錄

有用 AI 就寫下來，沒用就寫「沒有使用」。

| 用了哪個 AI | 問了什麼（大概就好） | 它哪裡幫到你 | 它哪裡說錯、你怎麼發現的 |
|---|---|---|---|
| （例：ChatGPT） | （例：dynamic_axes 是什麼意思） |  |  |

## 繳交前檢查

1. **重跑一次**：「執行階段 › 重新啟動工作階段並執行所有儲存格」，每一格都有輸出、沒有紅字（練習三之二那兩段錯誤訊息是用 try 接住故意印出來的，不算紅字）。
2. **檔名**：`深度學習_W15_學號_姓名.ipynb`（左上角點檔名就能改）。
3. **截圖**：5 張（①～⑤），每張都是整個螢幕，看得到身份指紋（左上的檔名、右上的 Google 頭貼、右下工作列的日期時間）和那一題的輸出。
4. **開放分享**：右上角「分享」→「一般存取權」改成「知道連結的任何人」→ 角色選「檢視者」→「複製連結」。
5. **自己檢查**：開一個無痕視窗貼上連結，不用登入就打得開才算成功。
6. **下載**：「檔案 › 下載 › 下載 .ipynb」。
7. **交到 Flipclass**：分享連結 1 個、.ipynb 檔 1 個、截圖 5 張（①～⑤），交到 Flipclass 作業區，期限照 Flipclass 規定。